In [0]:
catalog = "projeto_cinedata"
bronze_schema_name = "bronze"
silver_schema_name = "silver"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

============================================================================================= silver.tb_info_filmes (origem: tb_movies_info)

1. Visualização

In [0]:
df_info = spark.table(f"{bronze_schema}.tb_movies_info")

In [0]:
df_duplicatas = df_info.groupBy("id").count().filter("count > 1")

print(f"Quantidade de IDs únicos que possuem duplicatas: {df_duplicatas.count()}")

display(df_duplicatas.orderBy(F.col("count").desc()).limit(10))

In [0]:
display(df_info.groupBy("status").count().orderBy(F.col("count").desc()))

In [0]:
display(
    df_info.select("release_date")
    .filter(F.col("release_date").isNotNull())
    .groupBy("release_date")
    .count()
    .orderBy(F.col("count").desc())
    .limit(20)
)

2. Deduplicação: Manter exclusivamente a versão mais recente por id_filme

In [0]:
# Garante a unicidade dos filmes. Caso haja reprocessamentos ou múltiplas atualizações na camada Bronze,
# mantemos exclusivamente a versão mais recente baseada na data de ingestão (ingestion_datetime).
window_dedup = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())

df_dedup = (
    df_info
    .withColumn("row_num", F.row_number().over(window_dedup))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

3. Transformações: Renomear, Normalizar, Converter e Derivar

In [0]:
df_silver_info = (
    df_dedup
    # Renomeação: Traduz as colunas para português (PT-BR) para padronizar o vocabulário 
    # de dados com a área de negócios.
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("release_date", "data_lancamento")
    .withColumnRenamed("runtime", "duracao_minutos")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("status", "status_filme")
    .withColumnRenamed("overview", "sinopse")
    .withColumnRenamed("tagline", "frase_divulgacao")
    
    # Tratamento de Strings: Remove espaços em branco e padroniza a capitalização (Initcap)
    # para evitar que erros de digitação na origem criem categorias duplicadas (ex: "Released" e "RELEASED").
    .withColumn("status_norm", F.initcap(F.trim(F.col("status_filme"))))
    # Tratamento de Strings: Remove espaços em branco e padroniza a capitalização (Initcap)
    # para evitar que erros de digitação na origem criem categorias duplicadas (ex: "Released" e "RELEASED").
    .withColumn(
        "status_filme",
        F.when(F.col("status_norm") == "Released", "Lançado")
         .when(F.col("status_norm") == "Post Production", "Pós-Produção")
         .when(F.col("status_norm") == "In Production", "Em Produção")
         .when(F.col("status_norm") == "Planned", "Planejado")
         .when(F.col("status_norm") == "Rumored", "Rumores")
         .when(F.col("status_norm") == "Canceled", "Cancelado")
         .otherwise("Não Informado")
    )
    .drop("status_norm")
    # Tratamento de Datas: Resolve inconsistências oriundas da fonte dos dados, lidando com formatos variados 
    # ('yyyy-MM-dd' e 'dd/MM/yyyy') e garantindo que o tipo final seja uma Data válida e padronizada.
    .withColumn(
        "data_lancamento",
        F.coalesce(
            F.expr("try_to_date(data_lancamento, 'yyyy-MM-dd')"),
            F.expr("try_to_date(data_lancamento, 'dd/MM/yyyy')")
        )
    )
    
    # Derivação Estratégica: Extrai previamente o ano de lançamento para facilitar análises 
    # anuais (Year-over-Year) de catálogo sem exigir cálculos nas ferramentas de BI visual.
    .withColumn("ano_lancamento", F.year(F.col("data_lancamento")))
)

colunas_finais = [
    "id_filme", "titulo", "titulo_original", "data_lancamento", "ano_lancamento",
    "duracao_minutos", "idioma_original", "status_filme", "sinopse", 
    "frase_divulgacao", "ingestion_datetime"
]
df_silver_info = df_silver_info.select(*colunas_finais)

4. Salvar na camada Silver como Delta Table

In [0]:
(
    df_silver_info.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("projeto_cinedata.silver.tb_info_filmes")
)
print("[OK] Tabela silver.tb_info_filmes gerada com sucesso.")

5. Visualizar o resultado final

In [0]:
display(df_silver_info.limit(20))

============================================================================================= silver.tb_financeiro_filmes (origem: tb_movies_financials)

In [0]:
df_fin = spark.table(f"{bronze_schema}.tb_movies_financials")
df_dolar = spark.table(f"{bronze_schema}.tb_cotacao_dolar")

print(f"Total bruto: {df_fin.count()}")
df_fin.printSchema()

In [0]:
df_textos = df_fin.filter(F.col("budget").rlike("[a-zA-Z]") | F.col("revenue").rlike("[a-zA-Z]"))

print(f"Registros com letras/textos: {df_textos.count()}")
display(df_textos.groupBy("budget", "revenue").count().orderBy(F.col("count").desc()).limit(10))

In [0]:
df_amostra_num = df_fin.filter(~F.col("budget").rlike("[a-zA-Z]")).select("budget", "revenue")
display(df_amostra_num.limit(10))

1. Carregar as tabelas necessárias

In [0]:
df_fin = spark.table("projeto_cinedata.bronze.tb_movies_financials")
df_info = spark.table("projeto_cinedata.silver.tb_info_filmes")
df_dolar = spark.table("projeto_cinedata.bronze.tb_cotacao_dolar")

2. Deduplicação do Financeiro

In [0]:
win_dedup = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())
df_fin_dedup = df_fin.withColumn("rn", F.row_number().over(win_dedup)).filter("rn = 1").drop("rn")

3. Preparar Cotação

In [0]:
# REGRA DE NEGÓCIO: Preenchimento Contínuo de Cotação de Câmbio
# O mercado financeiro não opera em finais de semana ou feriados, mas filmes são lançados nesses dias.
# Utiliza-se Forward Fill (última cotação disponível) e Backward Fill (próxima cotação) para garantir 
# que todo filme tenha uma taxa de câmbio atrelada à sua data de lançamento para conversões precisas.
df_dolar_clean = df_dolar.withColumn("data_cotacao", F.expr("try_cast(dataHoraCotacao as date)"))

df_datas = df_silver_info.select(F.col("data_lancamento").alias("data_ref")).filter(F.col("data_ref").isNotNull()).distinct()
df_cotacoes = df_dolar_clean.select(F.col("data_cotacao").alias("data_ref"), "cotacaoCompra").distinct()

win_ffill = Window.partitionBy(F.lit(1)).orderBy("data_ref").rowsBetween(Window.unboundedPreceding, Window.currentRow)
win_bfill = Window.partitionBy(F.lit(1)).orderBy("data_ref").rowsBetween(Window.currentRow, Window.unboundedFollowing)

df_calendario_preenchido = (
    df_datas.join(df_cotacoes, "data_ref", "full_outer")
    .withColumn("cotacao_brl", F.last("cotacaoCompra", ignorenulls=True).over(win_ffill))
    .withColumn("cotacao_brl", F.first("cotacao_brl", ignorenulls=True).over(win_bfill))
    .filter(F.col("data_ref").isNotNull())
)

4. Join Base

In [0]:
df_base = (
    df_fin_dedup
    .join(df_silver_info.select("id_filme", "data_lancamento"), df_fin_dedup["id"] == df_silver_info["id_filme"], "inner")
    .drop(df_silver_info["id_filme"])
    .join(df_calendario_preenchido, F.col("data_lancamento") == F.col("data_ref"), "left")
)

5. Higienização das Moedas

In [0]:
# REGRA DE NEGÓCIO: Limpeza de Valores Financeiros Sujos
def higienizar_moeda(col_name):
    c = F.trim(F.col(col_name))
    # Nulifica placeholders textuais que indicam falta de dado, evitando que quebrem as agregações métricas.
    c = F.when(c.isin("Unknown", "Não Informado", "N/A", "null", "None"), F.lit(None)).otherwise(c)
    
    # Remove prefixos de moeda e caracteres especiais, mantendo apenas números, pontos, hifens e o sufixo "K".
    c = F.regexp_replace(c, r"[^0-9.K\-]", "")
    
    # Identifica valores sumarizados com a letra "K" (milhares) e os converte para a grandeza financeira real (x 1000).
    c_num = F.when(c.endswith("K"), F.regexp_replace(c, "K", "").cast("double") * 1000)\
             .otherwise(c.cast("double"))
             
    # Valores zerados ou negativos de custo/receita são considerados "falta de preenchimento" para a lógica do BI
    # e devem ser convertidos em Null para não distorcer médias.
    return F.when((c_num.isNotNull()) & (c_num > 0), c_num.cast("decimal(18,2)"))\
            .otherwise(F.lit(None))

df_silver_fin = (
    df_base
    .withColumnRenamed("id", "id_filme")
    # Aplicação da higienização nos campos brutos de orçamento e receita
    .withColumn("orcamento_usd", higienizar_moeda("budget"))
    .withColumn("receita_usd", higienizar_moeda("revenue"))
    
    # REGRA DE NEGÓCIO: Tropicalização de Valores[cite: 5]
    # Converte os valores em USD para BRL utilizando a cotação vinculada à data exata do lançamento.
    .withColumn("orcamento_brl", (F.col("orcamento_usd") * F.col("cotacao_brl")).cast("decimal(18,2)"))
    .withColumn("receita_brl", (F.col("receita_usd") * F.col("cotacao_brl")).cast("decimal(18,2)"))
    
    # REGRA DE NEGÓCIO: Criação de KPIs Financeiros Absolutos[cite: 5]
    # Calcula o lucro nominal em ambas as moedas (apenas se orçamento e receita forem conhecidos),
    # fornecendo métricas diretas de rentabilidade para a camada analítica.
    .withColumn(
        "lucro_usd",
        F.when(F.col("receita_usd").isNotNull() & F.col("orcamento_usd").isNotNull(),
               F.col("receita_usd") - F.col("orcamento_usd")
        ).otherwise(F.lit(None))
    )
    .withColumn(
        "lucro_brl",
        F.when(F.col("receita_brl").isNotNull() & F.col("orcamento_brl").isNotNull(),
               F.col("receita_brl") - F.col("orcamento_brl")
        ).otherwise(F.lit(None))
    )
    
    # REGRA DE NEGÓCIO: Criação de KPI de Eficiência Relativa[cite: 5]
    # Calcula a margem de lucro percentual, permitindo aos executivos comparar o sucesso 
    # de produções independentemente do tamanho do seu orçamento inicial.
    .withColumn(
        "margem_lucro_pct",
        F.when(
            (F.col("receita_usd").isNotNull()) & (F.col("receita_usd") > 0) & (F.col("lucro_usd").isNotNull()), 
            ((F.col("lucro_usd") / F.col("receita_usd")) * 100).cast("decimal(10,2)")
        ).otherwise(F.lit(None))
    )
)

In [0]:
F.when(F.col("orcamento_usd") < 1000, F.lit(None))

6. Organizar schema final e salvar

In [0]:
colunas_fin_finais = [
    "id_filme", "orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl",
    "lucro_usd", "lucro_brl", "margem_lucro_pct", "ingestion_datetime"
]

df_silver_fin_final = df_silver_fin.select(*colunas_fin_finais)

(
    df_silver_fin_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_financeiro_filmes")
)

display(df_silver_fin_final.limit(20))

============================================================================================= silver.tb_metricas_engajamento (origem: tb_movies_metrics)

In [0]:
df_bronze_metrics = spark.table(f"{bronze_schema}.tb_movies_metrics")

print(f"Total de registros brutos em movies_metrics: {df_bronze_metrics.count()}")
df_bronze_metrics.printSchema()

display(df_bronze_metrics.limit(20))

2. Mapeamento de colunas e Limpeza Inicial de Formatação

In [0]:
df_metrics = spark.table(f"{bronze_schema}.tb_movies_metrics")

df_silver_metrics = (
    df_metrics
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("popularity", "popularidade_raw")
    .withColumnRenamed("vote_average", "nota_media_tmdb_raw")
    .withColumnRenamed("vote_count", "qtd_votos_tmdb_raw")
    .withColumnRenamed("averageRating", "nota_media_imdb_raw")
    .withColumnRenamed("numVotes", "qtd_votos_imdb_raw")
)

In [0]:
def limpar_e_converter_decimal(col_name):
    query_sql = rf"TRY_CAST(REGEXP_REPLACE(REGEXP_REPLACE(TRIM({col_name}), ',', '.'), '[^0-9.\\-]', '') AS DECIMAL(10,4))"
    return F.expr(query_sql)

def limpar_e_converter_inteiro(col_name):
    query_sql = rf"TRY_CAST(REGEXP_REPLACE(TRIM({col_name}), '[^0-9\\-]', '') AS BIGINT)"
    return F.expr(query_sql)

3. Regras de Negócio de Limites e Escalas Numéricas

In [0]:
df_silver_metrics = (
    df_silver_metrics
    .withColumn("popularidade", limpar_e_converter_decimal("popularidade_raw"))
    .withColumn("nota_media_tmdb", limpar_e_converter_decimal("nota_media_tmdb_raw"))
    .withColumn("qtd_votos_tmdb", limpar_e_converter_inteiro("qtd_votos_tmdb_raw"))
    .withColumn("nota_media_imdb", limpar_e_converter_decimal("nota_media_imdb_raw"))
    .withColumn("qtd_votos_imdb", limpar_e_converter_inteiro("qtd_votos_imdb_raw"))
    
    .withColumn(
        "popularidade",
        F.when(F.col("popularidade") < 0, F.lit(None)).otherwise(F.col("popularidade"))
    )
    .withColumn(
        "nota_media_tmdb",
        F.when((F.col("nota_media_tmdb") >= 0) & (F.col("nota_media_tmdb") <= 10), F.col("nota_media_tmdb"))
         .otherwise(F.lit(None))
    )
    .withColumn(
        "qtd_votos_tmdb",
        F.when(F.col("qtd_votos_tmdb") < 0, F.lit(None)).otherwise(F.col("qtd_votos_tmdb"))
    )
    .withColumn(
        "nota_media_imdb",
        F.when((F.col("nota_media_imdb") >= 0) & (F.col("nota_media_imdb") <= 10), F.col("nota_media_imdb"))
         .otherwise(F.lit(None))
    )
    .withColumn(
        "qtd_votos_imdb",
        F.when(F.col("qtd_votos_imdb") < 0, F.lit(None)).otherwise(F.col("qtd_votos_imdb"))
    )
)

4. Deduplicação

In [0]:
window_dedup_metrics = Window.partitionBy("id_filme").orderBy(F.col("ingestion_datetime").desc())

df_silver_metrics_final = (
    df_silver_metrics
    .withColumn("rn", F.row_number().over(window_dedup_metrics))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

colunas_metrics_finais = [
    "id_filme", "popularidade", "nota_media_tmdb", "qtd_votos_tmdb",
    "nota_media_imdb", "qtd_votos_imdb", "ingestion_datetime"
]

df_silver_metrics_final = df_silver_metrics_final.select(*colunas_metrics_finais)

5. Persistência na Camada Silver

In [0]:
(
    df_silver_metrics_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_metricas_engajamento")
)

display(df_silver_metrics_final.limit(20))

============================================================================================= silver.tb_avaliacoes_usuarios (Origem: movies_reviews)

1. Visualização dos dados

In [0]:
df_bronze_reviews = spark.table(f"{bronze_schema}.tb_movies_reviews")

print(f"Total de registros brutos em movies_reviews: {df_bronze_reviews.count()}")
df_bronze_reviews.printSchema()

display(df_bronze_reviews.limit(20))

2. Mapeamento de colunas e limpeza inicial

In [0]:
df_reviews = spark.table(f"{bronze_schema}.tb_movies_reviews")

df_silver_reviews = (
    df_reviews
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("nome", "nome_usuario")
    .withColumnRenamed("nota", "nota_usuario")
    .withColumnRenamed("comentario", "comentario_usuario")
    
    .withColumn("nota_usuario", F.col("nota_usuario").cast("decimal(4,2)"))
    .withColumn("comentario_usuario", F.trim(F.col("comentario_usuario")))
    
    .withColumn(
        "nota_usuario",
        F.when((F.col("nota_usuario") >= 0) & (F.col("nota_usuario") <= 10), F.col("nota_usuario"))
         .otherwise(F.lit(None))
    )
    
    .withColumn(
        "comentario_usuario",
        F.when(
            (F.col("comentario_usuario").isNull()) | 
            (F.col("comentario_usuario") == "") | 
            (F.lower(F.col("comentario_usuario")).isin("null", "none", "nan", "n/a")), 
            F.lit("Sem comentário")
        )
        .otherwise(F.col("comentario_usuario"))
    )
)

3. Deduplicação integral

In [0]:
window_dedup_reviews = Window.partitionBy(
    "id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"
).orderBy(F.col("ingestion_datetime").desc())

df_silver_reviews_final = (
    df_silver_reviews
    .withColumn("rn", F.row_number().over(window_dedup_reviews))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

4. Seleção final das colunas estruturadas

In [0]:
colunas_reviews_finais = [
    "id_filme", "nome_usuario", "nota_usuario", 
    "comentario_usuario", "ingestion_datetime"
]

df_silver_reviews_final = df_silver_reviews_final.select(*colunas_reviews_finais)

(
    df_silver_reviews_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_avaliacoes_usuarios")
)

display(df_silver_reviews_final.limit(20))

============================================================================================= silver.tb_generos (origem: tb_credits_and_tags, coluna genres)

1. Seleção, Padronização de Delimitadores e Explode

In [0]:
df_tags = spark.table(f"{bronze_schema}.tb_credits_and_tags")

df_exploded_genres = (
    df_tags.select(
        F.col("id").alias("id_filme"),
        F.col("genres"),
        "ingestion_datetime"
    )
    .withColumn("genres_norm", F.regexp_replace(F.col("genres"), r"[;|]", ","))
    .withColumn("genero", F.explode_outer(F.split(F.col("genres_norm"), ",")))
    .withColumn("genero", F.trim(F.col("genero")))
)

2. Limpeza de Resíduos e Column Shift

In [0]:
generos_validos = [
    "Action", "Adventure", "Animation", "Comedy", "Crime", 
    "Documentary", "Drama", "Family", "Fantasy", "History", 
    "Horror", "Music", "Mystery", "Romance", "Science Fiction", 
    "Sci-Fi", "TV Movie", "Thriller", "War", "Western"
]

df_silver_generos = (
    df_exploded_genres
    .filter((F.col("genero").isNotNull()) & (F.col("genero") != ""))
    .filter(~F.lower(F.col("genero")).isin("n/a", "null", "none", "nan"))
    .filter(F.col("genero").isin(generos_validos))
)

3. Deduplicação (garante que um filme não tenha o mesmo gênero repetido)

In [0]:
window_dedup_genres = Window.partitionBy("id_filme", "genero").orderBy(F.col("ingestion_datetime").desc())

df_silver_generos_final = (
    df_silver_generos
    .withColumn("rn", F.row_number().over(window_dedup_genres))
    .filter(F.col("rn") == 1)
    .select("id_filme", "genero", "ingestion_datetime")
)

4. Persistência na camada Silver

In [0]:
(
    df_silver_generos_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_generos")
)

display(df_silver_generos_final.limit(15))

============================================================================================= silver.tb_pessoas_empresas (origem: tb_credits_and_tags, colunas cast, directors, writers,
production_companies)

In [0]:
df_bronze_tags = spark.table(f"{bronze_schema}.tb_credits_and_tags")

df_amostra_entidades = df_bronze_tags.select(
    "id", "cast", "directors", "writers", "production_companies"
)

display(df_amostra_entidades.limit(10))

1. Função para explodir e limpar cada coluna de entidade

In [0]:
df_tags = spark.table(f"{bronze_schema}.tb_credits_and_tags")

def extrair_entidade(df, col_name, tipo_entidade_nome):
    return (
        df.select(
            F.col("id").alias("id_filme"),
            F.col(col_name),
            "ingestion_datetime"
        )
        .withColumn("norm", F.regexp_replace(F.col(col_name), r"[;|]", ","))
        .withColumn("nome_entidade", F.explode_outer(F.split(F.col("norm"), ",")))
        .withColumn("nome_entidade", F.trim(F.col("nome_entidade")))
        .withColumn("nome_entidade", F.initcap(F.col("nome_entidade")))
        .withColumn("tipo_entidade", F.lit(tipo_entidade_nome))
        .select("id_filme", "nome_entidade", "tipo_entidade", "ingestion_datetime")
    )

2. Extrai cada tipo de entidade

In [0]:
df_atores = extrair_entidade(df_tags, "cast", "Ator")
df_diretores = extrair_entidade(df_tags, "directors", "Diretor")
df_roteiristas = extrair_entidade(df_tags, "writers", "Roteirista")
df_produtoras = extrair_entidade(df_tags, "production_companies", "Produtora")

3. Empilha todos os DataFrames

In [0]:
df_unificado = df_atores.union(df_diretores).union(df_roteiristas).union(df_produtoras)

4. Limpeza e Filtros de Resíduos

In [0]:
df_silver_pessoas = (
    df_unificado
    .filter((F.col("nome_entidade").isNotNull()) & (F.col("nome_entidade") != ""))
    .filter(~F.lower(F.col("nome_entidade")).isin("n/a", "null", "none", "nan"))
    .filter(~F.col("nome_entidade").rlike(r"^[0-9.\-]+$"))
    .filter(~F.col("nome_entidade").like("%.jpg"))
    .filter(F.length(F.col("nome_entidade")) <= 100)
)

5. Deduplicação

In [0]:
window_dedup_pessoas = Window.partitionBy(
    "id_filme", "nome_entidade", "tipo_entidade"
).orderBy(F.col("ingestion_datetime").desc())

df_silver_pessoas_final = (
    df_silver_pessoas
    .withColumn("rn", F.row_number().over(window_dedup_pessoas))
    .filter(F.col("rn") == 1)
    .select("id_filme", "nome_entidade", "tipo_entidade", "ingestion_datetime")
)

6. Gravação na Silver

In [0]:
(
    df_silver_pessoas_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_pessoas_empresas")
)

display(df_silver_pessoas_final.limit(15))

============================================================================================= silver.tb_cotacao_dolar (origem: tb_cotacao_dolar)

1. Padroniza a data e remove duplicatas diárias

In [0]:
df_bronze_cotacao = spark.table(f"{bronze_schema}.tb_cotacao_dolar")

df_cotacao_limpa = (
    df_bronze_cotacao
    .withColumn("data_ref", F.to_date(F.col("dataHoraCotacao")))
    .withColumn("cotacao_compra", F.col("cotacaoCompra").cast("decimal(10,4)"))
)

window_dia = Window.partitionBy("data_ref").orderBy(F.col("dataHoraCotacao").desc())

df_cotacao_diaria = (
    df_cotacao_limpa
    .withColumn("rn", F.row_number().over(window_dia))
    .filter(F.col("rn") == 1)
    .select("data_ref", "cotacao_compra")
)

2. Criação da Série Temporal Contínua

In [0]:
min_date, max_date = df_cotacao_diaria.select(F.min("data_ref"), F.max("data_ref")).first()

df_calendario = spark.sql(f"""
    SELECT explode(sequence(to_date('{min_date}'), to_date('{max_date}'), interval 1 day)) as data_ref
""")

3. Cruzamento e aplicação do Forward Fill

In [0]:
window_ffill = (
    Window
    .partitionBy(F.lit(1))
    .orderBy("data_ref")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_silver_cotacao_final = (
    df_calendario
    .join(df_cotacao_diaria, "data_ref", "left")
    .withColumn("cotacao_compra_ffill", F.last("cotacao_compra", ignorenulls=True).over(window_ffill))
    .select(
        "data_ref",
        F.col("cotacao_compra_ffill").alias("cotacao_compra")
    )
    .filter(F.col("cotacao_compra").isNotNull())
)

4. Persistência na camada Silver

In [0]:
(
    df_silver_cotacao_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_cotacao_dolar")
)

display(df_silver_cotacao_final.limit(15))